# Seasonal Upset Frequency Data 

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
HERE = Path.cwd()
REPO_ROOT = HERE.parents[3]

DATA_DIR = REPO_ROOT / "data" / "european_soccer_leagues" / "pure_skill"
OUT_CSV  = HERE / "season_upset_frequency_summary_all_seeds.csv"

LEAGUES = ["bundesliga", "la_liga", "premier_league", "serie_a"]

def load_files(league: str):
    matches = pd.read_csv(DATA_DIR / f"{league}_simulated_matches_all_seeds.csv")
    standings = pd.read_csv(DATA_DIR / f"{league}_simulated_standings_all_seasons.csv")
    return matches, standings

In [3]:
def compute_upset_freq(league: str) -> pd.DataFrame:
    matches, standings = load_files(league)
    seed_cols = [c for c in matches.columns if c.startswith("simulated_home_team_result_seed_")]
    seed_nums = sorted(int(c.split("_")[-1]) for c in seed_cols)

    out_rows = []

    for season, g in matches.groupby("season", sort=True):
        total_matches = len(g)
        rec = {"league": league, "season": int(season), "total_matches": total_matches}

        upset_counts, upset_freqs = [], []

        for n in seed_nums:
            # standings for this seed
            rank_col = f"simulated_rank_{n}"
            ranks = standings[standings["season"] == season][["team", rank_col]]

            # merge ranks into match data
            merged = (
                g.merge(ranks.rename(columns={"team": "home_team", rank_col: "home_rank"}), on="home_team")
                 .merge(ranks.rename(columns={"team": "away_team", rank_col: "away_rank"}), on="away_team")
            )

            sim_col = f"simulated_home_team_result_seed_{n}"
            res = merged[sim_col].to_numpy()
            home_rank = merged["home_rank"].to_numpy()
            away_rank = merged["away_rank"].to_numpy()

            # compute upsets
            upsets = np.zeros(len(res))
            upsets[(res == 1) & (home_rank > away_rank)] = 1.0
            upsets[(res == -1) & (away_rank > home_rank)] = 1.0
            upsets[(res == 0) & (home_rank != away_rank)] = 1/3.0

            total_upsets = upsets.sum()
            freq = total_upsets / total_matches if total_matches > 0 else np.nan

            rec[f"total_upsets_seed_{n}"] = total_upsets
            rec[f"upset_frequency_seed_{n}"] = freq
            upset_counts.append(total_upsets)
            upset_freqs.append(freq)

        # averages
        rec["total_upsets_avg"] = np.mean(upset_counts)
        rec["upset_frequency_avg"] = np.mean(upset_freqs)

        out_rows.append(rec)

    # column order
    pairs = []
    for n in seed_nums:
        pairs += [f"total_upsets_seed_{n}", f"upset_frequency_seed_{n}"]

    cols = ["league", "season", "total_matches"] + pairs + ["total_upsets_avg", "upset_frequency_avg"]
    return pd.DataFrame(out_rows)[cols]

In [4]:
frames = [compute_upset_freq(lg) for lg in LEAGUES]
combined = pd.concat(frames, ignore_index=True)
combined = combined.sort_values(["league", "season"]).reset_index(drop=True)

combined.to_csv(OUT_CSV, index=False)
print(f"✅ Upset frequency summary written to: {OUT_CSV}")
combined.head()

✅ Upset frequency summary written to: /Users/adhvik_rayaprolu/Desktop/uiuc/IML_Fall2025/skillvsluck/output/european_soccer_leagues/upset_frequency/pure_skill_updated/season_upset_frequency_summary_all_seeds.csv


,league,season,total_matches,total_upsets_seed_1,upset_frequency_seed_1,total_upsets_seed_2,upset_frequency_seed_2,total_upsets_seed_3,upset_frequency_seed_3,total_upsets_seed_4,...,total_upsets_seed_7,upset_frequency_seed_7,total_upsets_seed_8,upset_frequency_seed_8,total_upsets_seed_9,upset_frequency_seed_9,total_upsets_seed_10,upset_frequency_seed_10,total_upsets_avg,upset_frequency_avg
0,bundesliga,2004,306,99.333333,0.324619,125.666667,0.410675,126.333333,0.412854,112.333333,...,105.333333,0.344227,112.666667,0.368192,120.666667,0.394336,118.000000,0.385621,114.066667,0.372767
1,bundesliga,2005,306,111.666667,0.364924,111.000000,0.362745,115.666667,0.377996,110.000000,...,116.333333,0.380174,111.666667,0.364924,113.666667,0.371460,106.000000,0.346405,112.500000,0.367647
2,bundesliga,2006,306,104.000000,0.339869,105.666667,0.345316,113.000000,0.369281,108.666667,...,120.666667,0.394336,111.333333,0.363834,118.333333,0.386710,119.666667,0.391068,113.533333,0.371024
3,bundesliga,2007,306,122.666667,0.400871,123.000000,0.401961,122.000000,0.398693,114.000000,...,120.666667,0.394336,121.000000,0.395425,120.333333,0.393246,120.333333,0.393246,119.833333,0.391612
4,bundesliga,2008,306,118.000000,0.385621,113.000000,0.369281,120.333333,0.393246,105.666667,...,109.000000,0.356209,103.666667,0.338780,113.666667,0.371460,116.666667,0.381264,113.700000,0.371569


# Overall Upset Frequency Data 

In [5]:
def compute_league_summary(season_df: pd.DataFrame) -> pd.DataFrame:
    """
    Get league level summary totals and frequencies across all seasons.
    """
    seed_nums = sorted({
        int(c.split("_")[-1])
        for c in season_df.columns
        if c.startswith("upset_frequency_seed_")
    })

    records = []
    for lg, g in season_df.groupby("league", sort=True):
        rec = {"league": lg}
        total_matches = g["total_matches"].sum()
        rec["total_matches"] = int(total_matches)

        upset_counts, upset_freqs = [], []

        for n in seed_nums:
            tot_col = f"total_upsets_seed_{n}"
            freq_col = f"upset_frequency_seed_{n}"

            total_upsets = g[tot_col].sum()
            freq = total_upsets / total_matches if total_matches > 0 else np.nan

            rec[f"total_upsets_seed_{n}"] = total_upsets
            rec[f"upset_frequency_seed_{n}"] = freq

            upset_counts.append(total_upsets)
            upset_freqs.append(freq)

        rec["total_upsets_avg"] = np.mean(upset_counts)
        rec["upset_frequency_avg"] = np.mean(upset_freqs)

        records.append(rec)

    pairs = []
    for n in seed_nums:
        pairs += [f"total_upsets_seed_{n}", f"upset_frequency_seed_{n}"]

    cols = ["league", "total_matches"] + pairs + ["total_upsets_avg", "upset_frequency_avg"]
    return pd.DataFrame(records)[cols]

In [7]:
league_summary = compute_league_summary(combined)

OUT_CSV_LEAGUE = HERE / "overall_league_upset_frequency_summary_all_seeds.csv"
league_summary.to_csv(OUT_CSV_LEAGUE, index=False)

print(f"✅ League-level upset summary written to: {OUT_CSV_LEAGUE}")
league_summary

✅ League-level upset summary written to: /Users/adhvik_rayaprolu/Desktop/uiuc/IML_Fall2025/skillvsluck/output/european_soccer_leagues/upset_frequency/pure_skill_updated/overall_league_upset_frequency_summary_all_seeds.csv


,league,total_matches,total_upsets_seed_1,upset_frequency_seed_1,total_upsets_seed_2,upset_frequency_seed_2,total_upsets_seed_3,upset_frequency_seed_3,total_upsets_seed_4,upset_frequency_seed_4,...,total_upsets_seed_7,upset_frequency_seed_7,total_upsets_seed_8,upset_frequency_seed_8,total_upsets_seed_9,upset_frequency_seed_9,total_upsets_seed_10,upset_frequency_seed_10,total_upsets_avg,upset_frequency_avg
0,bundesliga,6426,2399.000000,0.373327,2437.666667,0.379344,2463.000000,0.383287,2392.000000,0.372238,...,2402.000000,0.373794,2400.000000,0.373483,2411.333333,0.375246,2438.333333,0.379448,2429.233333,0.378032
1,la_liga,8740,3395.666667,0.388520,3390.333333,0.387910,3337.666667,0.381884,3366.000000,0.385126,...,3341.666667,0.382342,3330.000000,0.381007,3360.666667,0.384516,3315.333333,0.379329,3343.700000,0.382574
2,premier_league,8360,3168.000000,0.378947,3199.666667,0.382735,3200.666667,0.382855,3223.666667,0.385606,...,3224.000000,0.385646,3213.333333,0.384370,3209.666667,0.383931,3158.000000,0.377751,3200.766667,0.382867
3,serie_a,8518,3220.666667,0.378101,3271.666667,0.384089,3257.000000,0.382367,3307.333333,0.388276,...,3256.000000,0.382249,3193.666667,0.374932,3273.666667,0.384323,3242.333333,0.380645,3255.666667,0.382210
